# Preparação de Dados — BORALI

Este notebook cobre as tasks de preparação de dados do projeto:
1. Definir fontes de dados
2. Normalizar dados
3. Exportar dataset tratado

In [1]:
import pandas as pd
import json
import re
from pathlib import Path

RAW = Path('../raw')
PROCESSED = Path('../processed')

---
## 1. Definir fontes de dados

| # | Fonte | Tipo | Descrição |
|---|-------|------|-----------|
| 1 | `eventos_culturais_recife_raw.csv` | Primária / Real | Formulário Google Forms aplicado a 47 respondentes sobre descoberta de eventos culturais no Recife (março 2026) |

O dataset cobre perfil demográfico, bairro de residência, frequência de participação em eventos, canais de descoberta, dificuldades enfrentadas e interesse em um aplicativo de recomendação — insumos diretos para o sistema de recomendação, mapa de eventos e análise de comportamento do usuário na plataforma BORALI.

In [2]:
df = pd.read_csv(RAW / 'eventos_culturais_recife_raw.csv', encoding='utf-8')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (45, 17)


,Timestamp,Qual é a sua idade?,Em qual bairro ou região do Recife você mora?,Com que frequência você participa de eventos culturais?,Que tipo de eventos culturais você costuma frequentar?,Onde você costuma descobrir eventos culturais?,Quantas plataformas diferentes você costuma usar para descobrir eventos?,Você considera fácil encontrar eventos culturais que combinem com seus gostos e que aconteçam perto de você?,Você já perdeu um evento que gostaria de ir porque descobriu tarde demais?,Você sente que eventos gratuitos ou comunitários têm pouca divulgação?,Você sente que existem poucos eventos divulgados no seu bairro ou região?,Você já foi a um evento que descobriu por acaso (sem ter buscado especificamente por ele)?,"Se um aplicativo recomendasse eventos automaticamente com base nos seus interesses, você acharia isso útil?",Você usaria um aplicativo que mostrasse eventos culturais próximos da sua localização?,Qual dessas funcionalidades você acharia mais útil?,Que outra funcionalidade você acha que seria útil em um aplicativo de eventos?,"Na sua opinião, o que mais dificulta descobrir eventos culturais no Recife?"
0,3/13/2026 7:47:56,16 - 24 anos,Espinheiro,Raramente (1 ou 2 vezes no ano),"Shows ou festivais musicais, Cinema, Eventos g...","Instagram, WhatsApp / grupos, Indicação de amigos",Nenhuma,Muito difícil,Sim,Concordo,Concordo totalmente,Sim,Muito útil,Com certeza usaria,"Eventos perto de mim, Recomendações baseadas n...",NaN,Informações espalhadas e dispersas
1,3/13/2026 8:19:46,16 - 24 anos,Casa Forte,Raramente (1 ou 2 vezes no ano),"Cinema, Eventos gastronômicos, Feiras culturai...",Indicação de amigos,Nenhuma,Difícil,Não lembro,Concordo totalmente,Concordo totalmente,Sim,Muito útil,Provavelmente usaria,"Eventos perto de mim, Recomendações baseadas n...",Alguma forma de interagir com os eventos: exem...,Falta de informação
2,3/13/2026 8:50:23,16 - 24 anos,Boa Viagem (Zona Sul),Nunca,Não costumo participar,"Indicação de amigos, Pesquisa no Google, Propa...",1,Difícil,Sim,Concordo,Concordo totalmente,Não,Neutro,Talvez usasse,"Eventos perto de mim, Recomendações baseadas n...",Uma função que indicasse algumas informações b...,"Uma melhor divulgação, especialmente, por meio..."


In [3]:
print('Colunas originais:')
for c in df.columns:
    print(' -', c)

Colunas originais:
 - Timestamp
 - Qual é a sua idade? 
 - Em qual bairro ou região do Recife você mora? 
 - Com que frequência você participa de eventos culturais? 
 - Que tipo de eventos culturais você costuma frequentar?  
 -   Onde você costuma descobrir eventos culturais?  
 - Quantas plataformas diferentes você costuma usar para descobrir eventos?  
 - Você considera fácil encontrar eventos culturais que combinem com seus gostos e que aconteçam perto de você?  
 - Você já perdeu um evento que gostaria de ir porque descobriu tarde demais? 
 - Você sente que eventos gratuitos ou comunitários têm pouca divulgação?  
 - Você sente que existem poucos eventos divulgados no seu bairro ou região? 
 -   Você já foi a um evento que descobriu por acaso (sem ter buscado especificamente por ele)?  
 - Se um aplicativo recomendasse eventos automaticamente com base nos seus interesses, você acharia isso útil?  
 -   Você usaria um aplicativo que mostrasse eventos culturais próximos da sua local

In [4]:
df.dtypes

Timestamp                                                                                                         object
Qual é a sua idade?                                                                                               object
Em qual bairro ou região do Recife você mora?                                                                     object
Com que frequência você participa de eventos culturais?                                                           object
Que tipo de eventos culturais você costuma frequentar?                                                            object
  Onde você costuma descobrir eventos culturais?                                                                  object
Quantas plataformas diferentes você costuma usar para descobrir eventos?                                          object
Você considera fácil encontrar eventos culturais que combinem com seus gostos e que aconteçam perto de você?      object
Você já perdeu um evento que gos

In [5]:
df.isnull().sum()

Timestamp                                                                                                          0
Qual é a sua idade?                                                                                                0
Em qual bairro ou região do Recife você mora?                                                                      0
Com que frequência você participa de eventos culturais?                                                            0
Que tipo de eventos culturais você costuma frequentar?                                                             0
  Onde você costuma descobrir eventos culturais?                                                                   0
Quantas plataformas diferentes você costuma usar para descobrir eventos?                                           0
Você considera fácil encontrar eventos culturais que combinem com seus gostos e que aconteçam perto de você?       0
Você já perdeu um evento que gostaria de ir porque descobriu tar

---
## 2. Normalizar dados

Etapas:
- Renomear colunas para snake_case
- Adicionar ID incremental
- Normalizar timestamp
- Padronizar faixa etária
- Padronizar bairro (title case)
- Normalizar frequência de participação
- Normalizar tipos de eventos (múltiplos valores → lista separada por `;`)
- Normalizar canais de descoberta (múltiplos valores → lista separada por `;`)
- Converter número de plataformas para numérico
- Padronizar facilidade de descoberta (escala ordinal)
- Padronizar campos booleanos
- Padronizar escalas Likert
- Normalizar funcionalidades desejadas (múltiplos valores → lista separada por `;`)

In [6]:
# Renomeia colunas
rename_map = {
    'Timestamp': 'timestamp',
    'Qual é a sua idade? ': 'faixa_etaria',
    'Em qual bairro ou região do Recife você mora? ': 'bairro',
    'Com que frequência você participa de eventos culturais? ': 'frequencia_eventos',
    'Que tipo de eventos culturais você costuma frequentar?  ': 'tipos_eventos',
    '  Onde você costuma descobrir eventos culturais?  ': 'canais_descoberta',
    'Quantas plataformas diferentes você costuma usar para descobrir eventos?  ': 'num_plataformas',
    'Você considera fácil encontrar eventos culturais que combinem com seus gostos e que aconteçam perto de você?  ': 'facilidade_descoberta',
    'Você já perdeu um evento que gostaria de ir porque descobriu tarde demais? ': 'perdeu_evento_tarde',
    'Você sente que eventos gratuitos ou comunitários têm pouca divulgação?  ': 'pouca_divulgacao_gratuitos',
    'Você sente que existem poucos eventos divulgados no seu bairro ou região? ': 'poucos_eventos_bairro',
    '  Você já foi a um evento que descobriu por acaso (sem ter buscado especificamente por ele)?  ': 'foi_evento_acaso',
    'Se um aplicativo recomendasse eventos automaticamente com base nos seus interesses, você acharia isso útil?  ': 'utilidade_recomendacao_app',
    '  Você usaria um aplicativo que mostrasse eventos culturais próximos da sua localização?  ': 'usaria_app_localizacao',
    'Qual dessas funcionalidades você acharia mais útil?  ': 'funcionalidades_uteis',
    'Que outra funcionalidade você acha que seria útil em um aplicativo de eventos? ': 'sugestao_funcionalidade',
    'Na sua opinião, o que mais dificulta descobrir eventos culturais no Recife? ': 'dificuldade_descoberta',
}
df = df.rename(columns=rename_map)
df.insert(0, 'id', range(1, len(df) + 1))
print('Colunas renomeadas e ID adicionado.')

Colunas renomeadas e ID adicionado.


In [7]:
# Normaliza timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
df['timestamp'] = df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S')
df['timestamp'].head()

0    2026-03-13T07:47:56
1    2026-03-13T08:19:46
2    2026-03-13T08:50:23
3    2026-03-13T08:51:54
4    2026-03-13T10:41:17
Name: timestamp, dtype: object

In [8]:
# Padroniza faixa etaria
faixa_map = {
    '16 - 24 anos': '16-24',
    '25 - 39 anos': '25-39',
    '40 - 59 anos': '40-59',
    '60 anos ou mais': '60+',
}
df['faixa_etaria'] = df['faixa_etaria'].map(faixa_map).fillna(df['faixa_etaria'])
df['faixa_etaria'].value_counts()

faixa_etaria
16-24    18
40-59    18
25-39     6
60+       3
Name: count, dtype: int64

In [9]:
# Padroniza bairro (title case e strip)
df['bairro'] = df['bairro'].str.strip().str.title()
df['bairro'].value_counts().head(10)

bairro
Boa Viagem                                                          5
Ibura                                                               5
Jardim São Paulo                                                    3
Candeias                                                            2
Moreno                                                              2
Jaboatão Dos Guararapes                                             2
Graças                                                              2
Não Moro Em Recife, Mas Por Vezes Participo De Eventos Na Região    1
Zona Sul/ Zona Norte/ Aldeia                                        1
Rosarinho                                                           1
Name: count, dtype: int64

In [10]:
# Padroniza frequencia de eventos
frequencia_map = {
    'Nunca': 'Nunca',
    'Raramente (1 ou 2 vezes no ano)': 'Raramente',
    'Ás vezes (algumas vezes no ano)': 'Às vezes',
    'Frequentemente (1 vez por mês)': 'Frequentemente',
    'Muito frequentemente (mais de 1 vez ao mês)': 'Muito frequentemente',
}
df['frequencia_eventos'] = df['frequencia_eventos'].map(frequencia_map).fillna(df['frequencia_eventos'])
df['frequencia_eventos'].value_counts()

frequencia_eventos
Às vezes                28
Raramente               11
Frequentemente           4
Nunca                    1
Muito frequentemente     1
Name: count, dtype: int64

In [11]:
# Normaliza tipos de eventos (multiplos valores -> lista separada por ;)
def normalize_list_field(raw):
    if pd.isna(raw):
        return ''
    parts = [p.strip() for p in raw.split(',') if p.strip()]
    return '; '.join(parts)

df['tipos_eventos'] = df['tipos_eventos'].apply(normalize_list_field)
df['canais_descoberta'] = df['canais_descoberta'].apply(normalize_list_field)
df['funcionalidades_uteis'] = df['funcionalidades_uteis'].apply(normalize_list_field)
df[['tipos_eventos', 'canais_descoberta', 'funcionalidades_uteis']].head(3)

,tipos_eventos,canais_descoberta,funcionalidades_uteis
0,Shows ou festivais musicais; Cinema; Eventos g...,Instagram; WhatsApp / grupos; Indicação de amigos,Eventos perto de mim; Recomendações baseadas n...
1,Cinema; Eventos gastronômicos; Feiras culturai...,Indicação de amigos,Eventos perto de mim; Recomendações baseadas n...
2,Não costumo participar,Indicação de amigos; Pesquisa no Google; Propa...,Eventos perto de mim; Recomendações baseadas n...


In [12]:
# Converte numero de plataformas para numerico
plat_map = {'Nenhuma': 0, '4 ou mais': 4}
df['num_plataformas'] = df['num_plataformas'].replace(plat_map)
df['num_plataformas'] = pd.to_numeric(df['num_plataformas'], errors='coerce')
df['num_plataformas'].value_counts().sort_index()

num_plataformas
0     5
1    13
2    20
3     5
4     2
Name: count, dtype: int64

In [13]:
# Padroniza facilidade de descoberta (escala ordinal)
facilidade_map = {
    'Muito difícil': 'Muito difícil',
    'Difícil': 'Difícil',
    'Nem fácil nem difícil': 'Neutro',
    'Fácil': 'Fácil',
    'Muito fácil': 'Muito fácil',
}
df['facilidade_descoberta'] = df['facilidade_descoberta'].map(facilidade_map).fillna(df['facilidade_descoberta'])
df['facilidade_descoberta'].value_counts()

facilidade_descoberta
Difícil          17
Neutro           16
Fácil             8
Muito difícil     2
Muito fácil       2
Name: count, dtype: int64

In [14]:
# Padroniza campos booleanos
df['foi_evento_acaso'] = df['foi_evento_acaso'].map({'Sim': True, 'Não': False}).fillna(df['foi_evento_acaso'])

# perdeu_evento_tarde tem 'Não lembro' tambem
perdeu_map = {'Sim': True, 'Não': False, 'Não lembro': None}
df['perdeu_evento_tarde'] = df['perdeu_evento_tarde'].map(perdeu_map)

df[['foi_evento_acaso', 'perdeu_evento_tarde']].value_counts(dropna=False)

foi_evento_acaso  perdeu_evento_tarde
True              True                   32
False             True                    6
True              NaN                     5
False             False                   1
True              False                   1
Name: count, dtype: int64

In [15]:
# Padroniza escalas Likert
likert_map = {
    'Discordo totalmente': 'Discordo totalmente',
    'Discordo': 'Discordo',
    'Neutro': 'Neutro',
    'Concordo': 'Concordo',
    'Concordo totalmente': 'Concordo totalmente',
}
df['pouca_divulgacao_gratuitos'] = df['pouca_divulgacao_gratuitos'].map(likert_map).fillna(df['pouca_divulgacao_gratuitos'])
df['poucos_eventos_bairro'] = df['poucos_eventos_bairro'].map(likert_map).fillna(df['poucos_eventos_bairro'])

utilidade_map = {
    'Poco útil': 'Pouco útil',
    'Neutro': 'Neutro',
    'Útil': 'Útil',
    'Muito útil': 'Muito útil',
}
df['utilidade_recomendacao_app'] = df['utilidade_recomendacao_app'].map(utilidade_map).fillna(df['utilidade_recomendacao_app'])

usaria_map = {
    'Não usaria': 'Não usaria',
    'Talvez usasse': 'Talvez',
    'Provavelmente usaria': 'Provavelmente sim',
    'Com certeza usaria': 'Com certeza sim',
}
df['usaria_app_localizacao'] = df['usaria_app_localizacao'].map(usaria_map).fillna(df['usaria_app_localizacao'])

print('Utilidade recomendacao:')
print(df['utilidade_recomendacao_app'].value_counts())
print('\nUsaria app:')
print(df['usaria_app_localizacao'].value_counts())

Utilidade recomendacao:
utilidade_recomendacao_app
Muito útil    25
Útil          15
Neutro         3
Pouco útil     2
Name: count, dtype: int64

Usaria app:
usaria_app_localizacao
Com certeza sim      25
Provavelmente sim    10
Talvez                9
Não usaria            1
Name: count, dtype: int64


In [16]:
# Visao geral do dataset normalizado
print(f'Shape final: {df.shape}')
df.dtypes

Shape final: (45, 18)


id                             int64
timestamp                     object
faixa_etaria                  object
bairro                        object
frequencia_eventos            object
tipos_eventos                 object
canais_descoberta             object
num_plataformas                int64
facilidade_descoberta         object
perdeu_evento_tarde           object
pouca_divulgacao_gratuitos    object
poucos_eventos_bairro         object
foi_evento_acaso                bool
utilidade_recomendacao_app    object
usaria_app_localizacao        object
funcionalidades_uteis         object
sugestao_funcionalidade       object
dificuldade_descoberta        object
dtype: object

In [17]:
df.head()

,id,timestamp,faixa_etaria,bairro,frequencia_eventos,tipos_eventos,canais_descoberta,num_plataformas,facilidade_descoberta,perdeu_evento_tarde,pouca_divulgacao_gratuitos,poucos_eventos_bairro,foi_evento_acaso,utilidade_recomendacao_app,usaria_app_localizacao,funcionalidades_uteis,sugestao_funcionalidade,dificuldade_descoberta
0,1,2026-03-13T07:47:56,16-24,Espinheiro,Raramente,Shows ou festivais musicais; Cinema; Eventos g...,Instagram; WhatsApp / grupos; Indicação de amigos,0,Muito difícil,True,Concordo,Concordo totalmente,True,Muito útil,Com certeza sim,Eventos perto de mim; Recomendações baseadas n...,NaN,Informações espalhadas e dispersas
1,2,2026-03-13T08:19:46,16-24,Casa Forte,Raramente,Cinema; Eventos gastronômicos; Feiras culturai...,Indicação de amigos,0,Difícil,None,Concordo totalmente,Concordo totalmente,True,Muito útil,Provavelmente sim,Eventos perto de mim; Recomendações baseadas n...,Alguma forma de interagir com os eventos: exem...,Falta de informação
2,3,2026-03-13T08:50:23,16-24,Boa Viagem (Zona Sul),Nunca,Não costumo participar,Indicação de amigos; Pesquisa no Google; Propa...,1,Difícil,True,Concordo,Concordo totalmente,False,Neutro,Talvez,Eventos perto de mim; Recomendações baseadas n...,Uma função que indicasse algumas informações b...,"Uma melhor divulgação, especialmente, por meio..."
3,4,2026-03-13T08:51:54,16-24,Rosarinho,Às vezes,Shows ou festivais musicais; Cinema; Exposiçõe...,Instagram; WhatsApp / grupos; Indicação de amigos,2,Neutro,True,Neutro,Concordo,True,Útil,Com certeza sim,Eventos perto de mim; Recomendações baseadas n...,C9mpra de ingressos pelo propio aplicativo par...,A falta de divulgação
4,5,2026-03-13T10:41:17,40-59,Boa Viagem,Às vezes,Shows ou festivais musicais; Teatro; Cinema,Instagram; Indicação de amigos; Pesquisa no Go...,1,Difícil,True,Concordo totalmente,Concordo totalmente,True,Muito útil,Com certeza sim,Eventos perto de mim; Recomendações baseadas n...,Que tenha a opção de selecionar eventos por pr...,Falta de divulgação


---
## 3. Exportar dataset tratado

In [18]:
out_csv = PROCESSED / 'eventos_culturais_recife_tratado.csv'
df.to_csv(out_csv, index=False, encoding='utf-8-sig')
print(f'CSV exportado: {out_csv}')

CSV exportado: ..\processed\eventos_culturais_recife_tratado.csv


In [19]:
out_json = PROCESSED / 'eventos_culturais_recife_tratado.json'
records = json.loads(df.to_json(orient='records', force_ascii=False))
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f'JSON exportado: {out_json}')

JSON exportado: ..\processed\eventos_culturais_recife_tratado.json


In [20]:
print(f'Registros exportados: {len(df)}')
print(f'Colunas: {list(df.columns)}')

Registros exportados: 45
Colunas: ['id', 'timestamp', 'faixa_etaria', 'bairro', 'frequencia_eventos', 'tipos_eventos', 'canais_descoberta', 'num_plataformas', 'facilidade_descoberta', 'perdeu_evento_tarde', 'pouca_divulgacao_gratuitos', 'poucos_eventos_bairro', 'foi_evento_acaso', 'utilidade_recomendacao_app', 'usaria_app_localizacao', 'funcionalidades_uteis', 'sugestao_funcionalidade', 'dificuldade_descoberta']
